# Phase 6 · Notebook 03 — Head-to-Head Across All Six Models

Brings together every numbered result CSV the project has produced:

| Phase | Model | File |
|---|---|---|
| 1 | spaCy `en_core_web_trf` | `../../results/phase1_results.csv` |
| 2 | HF `dslim/bert-base-NER` | `../../phase2_baseline_comparison/results/hf_results.csv` |
| 2 | Presidio (stock + CASE_NUMBER) | `../../phase2_baseline_comparison/results/presidio_results.csv` |
| 2 | RoBERTa fine-tuned on TAB | `../../phase2_baseline_comparison/results/finetuned_results.csv` |
| 6 | LegalBERT fine-tuned on TAB | `../results/legalbert_results.csv` |
| 6 | Ensemble (3-predictor voting) | `../results/ensemble_results.csv` |

> Run Notebooks 01 and 02 first — this notebook just plots their output.

---


## Setup


In [ ]:
import sys
sys.path.insert(0, "../../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


## Load every results CSV


In [ ]:
def safe_read(path, model_label):
    try:
        df = pd.read_csv(path)
        if "model" not in df.columns:
            df.insert(0, "model", model_label)
        return df
    except FileNotFoundError:
        print(f"⚠️  {path} not found — run the upstream notebook first.")
        return None


frames = []
# Phase 1 doesn't have a model column — assign one
p1 = safe_read("../../results/phase1_results.csv", "spacy_trf")
if p1 is not None:
    p1["model"] = "spacy_trf"
    frames.append(p1)

for path in [
    "../../phase2_baseline_comparison/results/hf_results.csv",
    "../../phase2_baseline_comparison/results/presidio_results.csv",
    "../../phase2_baseline_comparison/results/finetuned_results.csv",
    "../results/legalbert_results.csv",
    "../results/ensemble_results.csv",
]:
    df = safe_read(path, path.split("/")[-1].replace(".csv", ""))
    if df is not None:
        frames.append(df)

if not frames:
    raise SystemExit("No results found — run the upstream notebooks first.")

results = pd.concat(frames, ignore_index=True)
print("Models loaded:", sorted(results["model"].unique()))


## Overall F1 — six-way comparison


In [ ]:
order = [
    "spacy_trf", "hf_bert_base_ner",
    "presidio_stock", "presidio_plus_case_number",
    "roberta_finetuned_tab", "legalbert_finetuned_tab",
    "ensemble_v1",
]
labels = {
    "spacy_trf":                 "spaCy en_core_web_trf",
    "hf_bert_base_ner":          "HF bert-base-NER",
    "presidio_stock":            "Presidio (stock)",
    "presidio_plus_case_number": "Presidio + CASE_NUMBER",
    "roberta_finetuned_tab":     "RoBERTa fine-tuned",
    "legalbert_finetuned_tab":   "LegalBERT fine-tuned",
    "ensemble_v1":               "Ensemble (3-way)",
}
colors = {
    "spacy_trf":                 "#95a5a6",
    "hf_bert_base_ner":          "#3498db",
    "presidio_stock":            "#9b59b6",
    "presidio_plus_case_number": "#8e44ad",
    "roberta_finetuned_tab":     "#27ae60",
    "legalbert_finetuned_tab":   "#16a085",
    "ensemble_v1":               "#e74c3c",
}

overall = results[(results["mode"] == "partial") & (results["entity_type"] == "_ALL")]
overall = overall.set_index("model").reindex([m for m in order if m in overall.index]).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    [labels.get(m, m) for m in overall["model"]],
    overall["f1"] * 100,
    color=[colors.get(m, "#7f8c8d") for m in overall["model"]],
    edgecolor="white",
)
for bar, f1 in zip(bars, overall["f1"]):
    ax.text(f1 * 100 + 0.5, bar.get_y() + bar.get_height() / 2, f"{f1:.1%}",
            va="center", fontsize=10)
ax.set_xlim(0, 100)
ax.set_xlabel("Overall F1 (%) — partial match, DIRECT + QUASI")
ax.set_title("Six-Way Head-to-Head on TAB Test", fontweight="bold")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig("../../figures/phase6_overall_f1.png", dpi=120, bbox_inches="tight")
plt.show()


## F1 by entity type


In [ ]:
entity_order = ["PERSON", "ORG", "LOC", "DATETIME", "QUANTITY", "CODE", "DEM", "MISC"]
per_type = results[(results["mode"] == "partial") & (results["entity_type"].isin(entity_order))]

pivot = (per_type
         .pivot_table(index="entity_type", columns="model", values="f1")
         .reindex(entity_order))
pivot = pivot.reindex(columns=[m for m in order if m in pivot.columns])

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(entity_order))
width = 0.8 / len(pivot.columns)
for i, model in enumerate(pivot.columns):
    offset = (i - (len(pivot.columns) - 1) / 2) * width
    vals = pivot[model].fillna(0).values * 100
    ax.bar([xi + offset for xi in x], vals, width=width,
           color=colors.get(model, "#7f8c8d"),
           label=labels.get(model, model),
           edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(entity_order)
ax.set_ylabel("F1 (%)")
ax.set_title("F1 by Entity Type — All Six Models", fontweight="bold")
ax.set_ylim(0, 100)
ax.legend(loc="upper right", framealpha=0.9, fontsize=8, ncols=2)
fig.tight_layout()
fig.savefig("../../figures/phase6_f1_by_entity.png", dpi=120, bbox_inches="tight")
plt.show()


## Where the ensemble helps most


In [ ]:
best_single = "legalbert_finetuned_tab"   # Phase 6 best single model
contender   = "ensemble_v1"

if best_single in pivot.columns and contender in pivot.columns:
    delta = ((pivot[contender] - pivot[best_single]) * 100).sort_values(ascending=False)
    print(f"F1 lift, Ensemble vs {labels[best_single]}:")
    print("─" * 50)
    for et, d in delta.items():
        bar = "█" * max(0, int(d / 2))
        sign = "+" if d >= 0 else ""
        print(f"  {et:10s}  {sign}{d:5.1f} pp   {bar}")


## What the plots tell us (predictions to verify against the actual numbers)

Three things to look for in the plots above:

1. **LegalBERT beats RoBERTa on legal-specific labels** (PERSON, ORG, MISC) — that's the domain-pretraining payoff. If LegalBERT loses on a label, it's probably DATETIME or QUANTITY (which are domain-agnostic — pretraining doesn't help).

2. **The ensemble beats every single member** on most labels. The interesting question is *which* labels — if it beats by 2 F1 across the board, the members are diverse and the ensemble is doing real work; if the lift is only on one label (probably CODE, thanks to Presidio's regex), the members are too correlated.

3. **No model dominates everywhere.** Even the ensemble probably has at least one label where a single specialist beats it — usually CODE (Presidio's regex pass alone is near-perfect) or DEM (LegalBERT alone, no noise from other members). That's expected and pointable-to-able in the writeup.
